# Multi-step + memory

The API is stateless. Every turn you re-send the whole past — memory is what you choose to re-send.

*The model wakes with amnesia every call. A "conversation" is a trick you perform by re-reading
the whole transcript aloud each turn — and eviction is deciding what to stop reading.*

## 0. Setup

`cp ../.env.example ../.env`, add `DEEPINFRA_API_KEY`.

In [1]:
import os

from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))
client = OpenAI(
    api_key=os.environ["DEEPINFRA_API_KEY"],
    base_url="https://api.deepinfra.com/v1/openai",
)
MODEL = "meta-llama/Meta-Llama-3.1-70B-Instruct-Turbo"


def say(messages, max_tokens=100):
    r = client.chat.completions.create(model=MODEL, max_tokens=max_tokens, messages=messages)
    return r.choices[0].message.content.strip(), r.usage.prompt_tokens

## 1. The gap

Nothing carries over. Tell it a fact, ask in a new call, it's gone.

In [2]:
FACT = "My favourite tokenizer is BPE."
Q = {"role": "user", "content": "What is my favourite tokenizer?"}

reply, _ = say([{"role": "user", "content": FACT}])
amnesia, _ = say([Q])
print("second call ->", amnesia[:90])

second call -> Unfortunately, I'm a large language model, I don't have personal access to your preference


## 2. Memory = replaying the transcript

Append every turn and re-send it. That's all "the model remembers" ever means.

In [3]:
CHAT = [
    {"role": "user", "content": FACT},
    {"role": "assistant", "content": reply},
    {"role": "user", "content": "I am working through a curriculum of 18 LLM mechanisms."},
    {"role": "assistant", "content": "That sounds like a solid plan."},
    {"role": "user", "content": "Today I built tool calling."},
    {"role": "assistant", "content": "Great — tool calling is the basis for agents."},
]

full, cost_full = say([*CHAT, Q])
print("full history ->", full[:60], f"| prompt_tokens={cost_full}")

full history -> I remember! Your favourite tokenizer is Byte Pair Encoding ( | prompt_tokens=189


## 3. It grows forever — so you evict

Replaying everything hits the context limit and the bill. Two ways to shrink it, and they
differ in *what they lose*.

In [4]:
def window(chat, keep=2):
    """Cheapest: keep the last N turns. Anything older is simply gone."""
    return chat[-keep:]


def summarise(chat, keep=2):
    """Compress the old turns into a sentence; keep the recent ones verbatim."""
    older, recent = chat[:-keep], chat[-keep:]
    text, _ = say([
        {"role": "system",
         "content": "Summarise the conversation in one or two sentences. Keep every fact."},
        *older,
        {"role": "user", "content": "Summarise the above."},
    ])
    return [{"role": "system", "content": f"Conversation so far: {text}"}, *recent]


win, cost_win = say([*window(CHAT), Q])
summ, cost_summ = say([*summarise(CHAT), Q])

print(f"window(2)  {cost_win:>4} tok ->", win[:60])
print(f"summarise  {cost_summ:>4} tok ->", summ[:60])
print(f"full       {cost_full:>4} tok ->", full[:60])

window(2)    42 tok -> I don't have any information about your preferences, includi
summarise    72 tok -> You've covered 5 mechanisms so far, including tokenizers. Ba
full        189 tok -> I remember! Your favourite tokenizer is Byte Pair Encoding (


## Weaknesses

| Weakness | What happens | Fix |
|---|---|---|
| **Windowing silently drops facts** | `window(2)` → "I don't have any information"; evicted context leaves no trace | Summarise, or keep more turns |
| **A second model picks what survives** | The summariser's judgement of "important", not yours | Prompt it for the facts you need |
| **Summarising isn't free** | full `189` tok, `summarise` `85`, `window(2)` `42` — but the summary cost ~100 tok to make | Only evict once the transcript dwarfs it |
| **Memory dies with the process** | `CHAT` is a Python list; restart = amnesia | Key by thread id in a store |
| **The transcript is the whole state** | Tool results share the list — eviction can drop what the next step needs | Evict by role, not position |